# 🧱 Lab: Fix the Broken Repo (4 Seeded Bugs)

**Module 5: Debugging Lab 1: Read the Failure** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Reproduce** every failure in an unfamiliar repo from a clean starting copy
2. **Localise** each bug to the file (pipeline stage) where it lives, using the debugging decision tree
3. **Fix** four seeded bugs, one change at a time, until `verify.py` is green from a fresh copy
4. **Write** a one-sentence cause and fix for each bug, then recall error → cause → fix for five more errors

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| The debugging decision tree | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | mini-walk-debug-tree |
| Silent failure | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. | mini-fix-data-failures |
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. | lab-clone-and-run-check |

## Prerequisites

- **mini-walk-debug-tree**: The printable decision tree
- **mini-fix-import-failures**: Import and environment failures
- **mini-fix-data-failures**: Silent data-shaped failures

## Scenario

You've just been handed a small Python repository — `broken_repo/` — that processes sales data. The previous developer left in a hurry. The repo has **four seeded bugs**: some crash loudly, some fail silently.

Your job: walk each failure through the debug tree and fix it, one bug at a time, until the verification script passes all checks.

| File | Pipeline stage |
|------|----------------|
| `loader.py` | Load: returns the sales records |
| `analysis.py` | Analyse: total revenue and top product |
| `report.py` | Report: formats the final summary |
| `verify.py` | Not a stage: runs 5 checks and prints PASS/FAIL for each |

## Step 0 — Create the Broken Repo

In a real project you would `git clone` the repo. Here the notebook writes the same files for you. The cell deletes any leftover copy first, so every run starts from the same clean state a fresh clone gives you: nothing from an earlier attempt props the code up.

The files carry no hints about where the bugs are. Finding them is the job.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import display, Markdown

# Start from a clean copy every time, like a fresh clone
repo_dir = "broken_repo"
shutil.rmtree(repo_dir, ignore_errors=True)
repo = Path(repo_dir)
repo.mkdir()

FILES = {
    "loader.py": '''"""Load sales data from a CSV-like source."""

from collections import OrderableDict


def load_sales():
    """Return a list of sales records (simulating a CSV read)."""
    raw_rows = [
        {"product": "Widget", "quantity": "15", "unit_price": 9.99},
        {"product": "Gadget", "quantity": "8",  "unit_price": 24.50},
        {"product": "Widget", "quantity": "22", "unit_price": 9.99},
        {"product": "Gizmo",  "quantity": "5",  "unit_price": 49.99},
        {"product": "Gadget", "quantity": "12", "unit_price": 24.50},
    ]
    return raw_rows
''',
    "analysis.py": '''"""Compute sales metrics."""


def total_revenue(sales):
    """Return the total revenue across all sales."""
    return sum(row["quantity"] * row["unit_price"] for row in sales)


def top_product(sales):
    """Return the product name with the highest total revenue."""
    revenue_by_product = {}
    for row in sales:
        name = row["product"]
        revenue_by_product[name] = revenue_by_product.get(name, 0) + (
            row["quantity"] * row["unit_price"]
        )
    return min(revenue_by_product, key=revenue_by_product.get)
''',
    "report.py": '''"""Format the final sales report."""


def build_report(total, top):
    """Return a formatted report string."""
    lines = [
        "=== Sales Report ===",
        f"Total Revenue: ${total:,.2f}",
        f"Top Product:   {top_product_name}",
        "===================",
    ]
    return "\\n".join(lines)
''',
}

for name, code in FILES.items():
    (repo / name).write_text(code, encoding="utf-8")
    print(f"Wrote {repo_dir}/{name}")

Wrote broken_repo/loader.py
Wrote broken_repo/analysis.py
Wrote broken_repo/report.py


`verify.py` is the repo's checker: five checks, each printed as PASS or FAIL. Treat it like a test suite a teammate wrote. You run it after every change; you never edit it to make a check pass.

In [2]:
VERIFY = '''"""Run 5 checks and print PASS/FAIL for each."""
EXPECTED_REVENUE = (15 + 22) * 9.99 + (8 + 12) * 24.50 + 5 * 49.99
results = []


def check(name, test):
    try:
        ok, detail = test()
    except Exception as e:
        # keep the message, drop the file path Python appends to import errors
        ok, detail = False, f"{type(e).__name__}: {str(e).split(' (')[0]}"
    results.append(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + ("" if ok else f"  —  {detail}"))


def sales():
    from loader import load_sales
    return load_sales()


def revenue():
    from analysis import total_revenue
    return total_revenue(sales())


def top():
    from analysis import top_product
    return top_product(sales())


def report():
    from report import build_report
    return build_report(revenue(), top())


check("loader imports", lambda: (len(sales()) == 5, "expected 5 rows"))
check("quantities are int", lambda: (all(isinstance(r["quantity"], int) for r in sales()), f"got {type(sales()[0]['quantity']).__name__}"))
check("total revenue correct", lambda: (abs(revenue() - EXPECTED_REVENUE) < 0.01, f"expected {EXPECTED_REVENUE:.2f}, got {revenue():.2f}"))
check("top product correct", lambda: (top() == "Gadget", f"expected Gadget, got {top()}"))
check("report builds correctly", lambda: ("Gadget" in report() and f"{EXPECTED_REVENUE:,.2f}" in report(), "report is missing the total or the top product"))

print()
print(f"{sum(results)}/{len(results)} checks passed")
print("✅ All green — repo is fixed!" if all(results) else "❌ Keep going — some checks still fail.")
'''

(repo / "verify.py").write_text(VERIFY, encoding="utf-8")
print(f"Wrote {repo_dir}/verify.py")

Wrote broken_repo/verify.py


## Helpers and the Debug Log

Four small helpers keep every bug fix short:

- `run_verify()` runs `verify.py` inside a repo folder, in a fresh Python process
- `show(file)` prints one repo file with line numbers
- `patch(file, old, new)` makes exactly **one** change to one file, and fails loudly if `old` is not there, so a typo can't silently do nothing
- `md(text)` renders a markdown table

The `debug_log` list collects one sentence of cause and one of fix for each bug.

In [3]:
# Child processes inherit the kernel's encoding. On Windows Jupyter that is often
# cp1252, which cannot print ✅ / ❌. Force UTF-8 for every repo command.
_SUBPROCESS_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}


def run_verify(cwd=repo_dir):
    """Run verify.py inside a repo folder and print its output."""
    result = subprocess.run(
        [sys.executable, "verify.py"],
        cwd=cwd,
        capture_output=True,
        text=True,
        encoding="utf-8",
        env=_SUBPROCESS_ENV,
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    return result.returncode


def show(filename):
    """Print one repo file with line numbers."""
    lines = (repo / filename).read_text(encoding="utf-8").splitlines()
    for number, line in enumerate(lines, 1):
        print(f"{number:>3}  {line}")


def patch(filename, old, new):
    """Make ONE change to one repo file; fail loudly if the old text is missing."""
    path = repo / filename
    code = path.read_text(encoding="utf-8")
    assert old in code, f"{old!r} not found in {filename}"
    path.write_text(code.replace(old, new, 1), encoding="utf-8")


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


debug_log = []

## Step 1 — Run the Verification Script (Baseline)

Before fixing anything, run the checker to see what fails. This is the baseline, and it is also the **Reproduce** step for the first bug.

In [4]:
run_verify()

  [FAIL] loader imports  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] quantities are int  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] total revenue correct  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] top product correct  —  ImportError: cannot import name 'OrderableDict' from 'collections'
  [FAIL] report builds correctly  —  ImportError: cannot import name 'OrderableDict' from 'collections'

0/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 1 — `loader.py` Won't Import (Loud Failure)

**Reproduce.** The baseline run above already did it: every check fails with the same `ImportError`. One loud bug at the top blocks everything downstream.

**Localise.** The first failing check is `loader imports`, and the message names the import: `from collections import OrderableDict` in `loader.py`, the loading stage.

**Shrink.** Already minimal: one import line.

**Change one thing.** Ask `collections` which names it really has, fix that one name, and re-verify. The import isn't even used yet, but a name that doesn't exist still stops the whole file from loading.

In [5]:
import collections
print([name for name in dir(collections) if "dict" in name.lower() and not name.startswith("_")])

patch("loader.py", "OrderableDict", "OrderedDict")   # the ONE change
debug_log.append({
    "bug": "Bug 1",
    "error": "ImportError: cannot import name 'OrderableDict'",
    "cause": "The import has a typo: collections has no OrderableDict.",
    "fix": "Changed the import to OrderedDict.",
})
run_verify()

['OrderedDict', 'UserDict', 'defaultdict']
  [PASS] loader imports
  [FAIL] quantities are int  —  got str
  [FAIL] total revenue correct  —  TypeError: can't multiply sequence by non-int of type 'float'
  [FAIL] top product correct  —  TypeError: can't multiply sequence by non-int of type 'float'
  [FAIL] report builds correctly  —  TypeError: can't multiply sequence by non-int of type 'float'

1/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 2 — Quantities Are Strings (Silent Failure)

**Reproduce.** `loader imports` passes now, but `quantities are int` fails with `got str`, and the next three checks crash with a `TypeError`. Nothing in `loader.py` raised an error: the wrong type travelled silently until the arithmetic tripped on it.

**Localise.** The values are already wrong when they leave the loading stage, so look at `loader.py`.

In [6]:
show("loader.py")

  1  """Load sales data from a CSV-like source."""
  2  
  3  from collections import OrderedDict
  4  
  5  
  6  def load_sales():
  7      """Return a list of sales records (simulating a CSV read)."""
  8      raw_rows = [
  9          {"product": "Widget", "quantity": "15", "unit_price": 9.99},
 10          {"product": "Gadget", "quantity": "8",  "unit_price": 24.50},
 11          {"product": "Widget", "quantity": "22", "unit_price": 9.99},
 12          {"product": "Gizmo",  "quantity": "5",  "unit_price": 49.99},
 13          {"product": "Gadget", "quantity": "12", "unit_price": 24.50},
 14      ]
 15      return raw_rows


`load_sales()` returns the rows exactly as a CSV reader would give them: `quantity` is text such as `"15"`.

**Shrink.** One record is enough to show it: `{"quantity": "15"}`.

**Change one thing.** Convert `quantity` to `int` as the rows leave the loader.

In [7]:
patch(
    "loader.py",
    "    return raw_rows",
    '    return [{**row, "quantity": int(row["quantity"])} for row in raw_rows]',
)   # the ONE change
debug_log.append({
    "bug": "Bug 2",
    "error": "Silent: quantities are str, so later arithmetic breaks",
    "cause": "The loader returned the CSV text values without converting them.",
    "fix": "Converted quantity to int as each row leaves load_sales().",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [FAIL] top product correct  —  expected Gadget, got Gizmo
  [FAIL] report builds correctly  —  NameError: name 'top_product_name' is not defined

3/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 3 — Wrong Top Product (Silent Failure)

**Reproduce.** Revenue is correct now, but `top product correct` says `expected Gadget, got Gizmo`. No exception, just a confident, wrong answer.

**Localise.** The loaded numbers are right (revenue passes), so the wrong answer is born in the analysis stage: `top_product` in `analysis.py`.

In [8]:
show("analysis.py")

  1  """Compute sales metrics."""
  2  
  3  
  4  def total_revenue(sales):
  5      """Return the total revenue across all sales."""
  6      return sum(row["quantity"] * row["unit_price"] for row in sales)
  7  
  8  
  9  def top_product(sales):
 10      """Return the product name with the highest total revenue."""
 11      revenue_by_product = {}
 12      for row in sales:
 13          name = row["product"]
 14          revenue_by_product[name] = revenue_by_product.get(name, 0) + (
 15              row["quantity"] * row["unit_price"]
 16          )
 17      return min(revenue_by_product, key=revenue_by_product.get)


`top_product` adds up the revenue per product correctly, then picks the winner with `min(...)`: the **lowest** revenue, not the highest.

**Shrink.** Two products with different revenues are enough for `min` to pick the wrong one.

**Change one thing.** Swap `min` for `max`.

In [9]:
patch("analysis.py", "return min(", "return max(")   # the ONE change
debug_log.append({
    "bug": "Bug 3",
    "error": "Silent: top product is Gizmo instead of Gadget",
    "cause": "top_product used min(), which returns the lowest-revenue product.",
    "fix": "Changed min() to max().",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [FAIL] report builds correctly  —  NameError: name 'top_product_name' is not defined

4/5 checks passed
❌ Keep going — some checks still fail.



0

---

## Bug 4 — Report Crashes with `NameError` (Loud Failure)

**Reproduce.** Four checks pass. `report builds correctly` fails with `NameError: name 'top_product_name' is not defined`.

**Localise.** The failing check is the reporting stage, and that name appears in one place: the f-string in `report.py`. The function's parameter is called `top`.

**Shrink.** Already minimal: one call to `build_report` triggers it.

**Change one thing.** Use the parameter name in the f-string.

In [10]:
patch("report.py", "{top_product_name}", "{top}")   # the ONE change
debug_log.append({
    "bug": "Bug 4",
    "error": "NameError: name 'top_product_name' is not defined",
    "cause": "The f-string used an old variable name instead of the parameter top.",
    "fix": "Changed {top_product_name} to {top}.",
})
run_verify()

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [PASS] report builds correctly

5/5 checks passed
✅ All green — repo is fixed!



0

## Final Verification — From a Fresh Copy

Green in your own folder is not the finish line. Copy the fixed repo to a new folder, the way a teammate gets it from a clone, and run `verify.py` there. All 5 checks should pass with nothing carried over from your session.

In [11]:
clone_dir = "broken_repo_clone"
shutil.rmtree(clone_dir, ignore_errors=True)
shutil.copytree(repo_dir, clone_dir, ignore=shutil.ignore_patterns("__pycache__"))

return_code = run_verify(cwd=clone_dir)

  [PASS] loader imports
  [PASS] quantities are int
  [PASS] total revenue correct
  [PASS] top product correct
  [PASS] report builds correctly

5/5 checks passed
✅ All green — repo is fixed!



## The Debug Log

One sentence of cause and one of fix for each bug. Then the cell cleans up both demo folders.

In [12]:
rows = "| Bug | Error | Cause | Fix |\n|-----|-------|-------|-----|\n"
for entry in debug_log:
    rows += f"| {entry['bug']} | {entry['error']} | {entry['cause']} | {entry['fix']} |\n"
md(rows)

shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(clone_dir, ignore_errors=True)
print(f"Cleaned up {repo_dir}/ and {clone_dir}/")

| Bug | Error | Cause | Fix |
|-----|-------|-------|-----|
| Bug 1 | ImportError: cannot import name 'OrderableDict' | The import has a typo: collections has no OrderableDict. | Changed the import to OrderedDict. |
| Bug 2 | Silent: quantities are str, so later arithmetic breaks | The loader returned the CSV text values without converting them. | Converted quantity to int as each row leaves load_sales(). |
| Bug 3 | Silent: top product is Gizmo instead of Gadget | top_product used min(), which returns the lowest-revenue product. | Changed min() to max(). |
| Bug 4 | NameError: name 'top_product_name' is not defined | The f-string used an old variable name instead of the parameter top. | Changed {top_product_name} to {top}. |


Cleaned up broken_repo/ and broken_repo_clone/


---
## Error → Cause → Fix

Above, you had the code in front of you and time to investigate. In real work, an error flashes on screen and you want the likely cause straight away. That instant recognition comes from **retrieval practice**: see an error, recall the cause and the fix, then check yourself.

For each of the five errors below: read it, say the cause, the fix, and whether it is loud or silent, **then** open the answer.

### Drill 1

```
ModuleNotFoundError: No module named 'pandas'
```

<details><summary>Answer</summary>

**Loud.** The package isn't installed in the environment this notebook runs in, or the wrong interpreter is selected. Fix: `uv add pandas` (or `uv sync` if it is already in `pyproject.toml`), then restart the kernel. Check `sys.executable` first: it should point into the project's `.venv`.

</details>

### Drill 2

```
ImportError: cannot import name 'soft_max' from 'torch.nn.functional'
```

<details><summary>Answer</summary>

**Loud.** The module exists, but that name doesn't: a typo, or the API changed between versions. Fix: check `dir(module)` or the docs for the real name (`softmax`).

</details>

### Drill 3

```python
prices = [{"item": "A", "price": "50"}, {"item": "B", "price": "120"}]
expensive = [p for p in prices if p["price"] > "80"]
print(len(expensive))  # prints: 0, but item B costs 120
```

<details><summary>Answer</summary>

**Silent.** The prices are strings and the threshold was quoted too, so Python compares text: `"120" < "80"` because `"1"` comes before `"8"`. Fix: compare numbers, `int(p["price"]) > 80`.

</details>

### Drill 4

```python
def midpoint(low, high):
    return low + high / 2    # meant: the value halfway between low and high

print(midpoint(0, 100))   # prints: 50.0  ← looks right
print(midpoint(40, 60))   # prints: 70.0  ← should be 50.0
```

<details><summary>Answer</summary>

**Silent.** Operator precedence: `/` runs before `+`, so only `high` is halved. With `low = 0` the bug is invisible, which is why the first test passed. Fix: `(low + high) / 2`. Test with inputs where the wrong logic and the right logic give different numbers.

</details>

### Drill 5

```
AttributeError: module 'email' has no attribute 'mime'
```

You have a file called `email.py` in your project folder.

<details><summary>Answer</summary>

**Loud, but misleading.** Your `email.py` shadows the standard library `email` package, so the error blames the wrong module. Fix: rename or delete your `email.py`, clear `__pycache__`, and restart the kernel. Check `email.__file__`: if it points into your project, you have a shadowing problem.

</details>

## 🎯 Summary

### What You Built

You repaired an unfamiliar repo with four seeded bugs — two loud crashes and two silent failures — by walking each one through the debugging decision tree until `verify.py` passed from a fresh copy, then drilled error → cause → fix on five more common failures.

### Key Takeaways

1. **Start with the verification script** — running it first shows which checks fail and which bug to fix first
2. **Loud failures first** — an import error at the top blocks every check downstream, so fix it before hunting silent bugs
3. **Silent failures need a known-good answer** — string quantities and `min` instead of `max` only showed up when output was compared with expected values
4. **One change at a time, re-verify** — re-running `verify.py` after each fix confirms it and reveals the next bug cleanly
5. **Prove it from a fresh copy** — the repo is only fixed when a clean copy passes the same checks with nothing carried over
6. **Error → cause → fix recall** — a short debug log and retrieval practice turn each failure into a pattern you recognise next time